# 07b — V3 补跑 (full Hetionet Node2Vec via PyG)

nb07 attempted a four-variant ablation of the walk graph.  V1 / V2 / V4 finished under `QUICK_RUN=True` (walk_length=20, num_walks=5).  **V3 (full Hetionet, ~47k nodes, ~2.25M edges) timed out** on CPU gensim — the Word2Vec fit alone didn't finish in one hour.

This notebook picks V3 up with **PyG's own `Node2Vec` implementation on MPS** (Apple GPU).  Same hyperparameters as nb07's QUICK_RUN to stay comparable across variants:

- `dimensions = 64, p = q = 1, walk_length = 20, num_walks = 5, window = 10`
- All 11 Hetionet node kinds kept
- Hub correction (top-20 Gene by degree) kept
- Test CtD edges dropped before walking
- LCC restriction after hub removal

Everything else — locked pair universe from nb04, scoring pipeline (Hadamard+LR, Concat+LR, Cosine), bootstrap 95% CI — is identical.

## 1 · Setup

In [1]:
from __future__ import annotations
import json, pathlib, pickle, time
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

import torch
from torch_geometric.nn import Node2Vec as PyGNode2Vec

from utils import load_hetnet, setup_plot_style
setup_plot_style()

ART   = pathlib.Path("artifacts")
CACHE = pathlib.Path("cache/node2vec"); CACHE.mkdir(parents=True, exist_ok=True)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Device: {device}")


Device: mps


In [2]:
# Locked split
meta  = json.loads((ART / "splits" / "split_meta.json").read_text())
pairs = pd.read_parquet(ART / "splits" / "pairs.parquet")
split = np.load(ART / "splits" / "lr_split.npz")
idx_train, idx_test = split["idx_train"], split["idx_test"]
SEED = int(meta["split_seed"])
y = pairs["y"].to_numpy()
pos_mask  = y == 1
test_mask = np.zeros(len(pairs), dtype=bool); test_mask[idx_test] = True
test_positives  = pairs.loc[pos_mask & test_mask,  ["compound", "disease"]]

torch.manual_seed(SEED)
np.random.seed(SEED)

# nb07's QUICK_RUN hyperparameters
DIMENSIONS  = 64
P, Q        = 1, 1
WALK_LENGTH = 20
NUM_WALKS   = 5
WINDOW      = 10
HUB_K       = 20


## 2 · Build V3 = full Hetionet + hub correction

Reuse the exact recipe from nb07's `build_v3_full`.  On MPS we can afford *all 11 node kinds* — the gensim CPU limitation is no longer the constraint.

In [3]:
t0 = time.time()
hetnet = load_hetnet()
print(f"Loaded hetnet in {time.time() - t0:.1f}s")

# All node kinds (no filter)
G = nx.Graph()
for n in hetnet["nodes"]:
    G.add_node((n["kind"], n["identifier"]), kind=n["kind"], name=n["name"])
for e in hetnet["edges"]:
    u, v = tuple(e["source_id"]), tuple(e["target_id"])
    G.add_edge(u, v, kind=e["kind"])

# Drop test CtD edges
dropped = 0
for c, d in test_positives.itertuples(index=False):
    u, v = ("Compound", c), ("Disease", d)
    if G.has_edge(u, v):
        G.remove_edge(u, v); dropped += 1

# Hub correction (top-K Gene by degree)
gene_deg = sorted(((n, deg) for n, deg in G.degree() if n[0] == "Gene"), key=lambda x: -x[1])
G.remove_nodes_from([n for n, _ in gene_deg[:HUB_K]])
lcc_nodes = max(nx.connected_components(G), key=len)
G = G.subgraph(lcc_nodes).copy()

print(f"V3 walk graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges, "
      f"{dropped} test CtD edges dropped.")


Loaded hetnet in 11.8s


V3 walk graph: 45,114 nodes, 2,069,117 edges, 151 test CtD edges dropped.


## 3 · PyG `Node2Vec` on MPS

PyG's `Node2Vec` implements the same biased random walks + Skip-gram objective as `eliorc/node2vec` (which nb05/07 use via `gensim`), but the whole pipeline lives inside PyTorch — walks are sampled with `torch_cluster.random_walk` and Skip-gram training runs on the GPU.  This is what makes the full-Hetionet run tractable.

In [4]:
node_list = list(G.nodes())
node_idx  = {n: i for i, n in enumerate(node_list)}

# Directed edge_index for PyG
src, dst = [], []
for u, v in G.edges():
    src.extend([node_idx[u], node_idx[v]])
    dst.extend([node_idx[v], node_idx[u]])
edge_index = torch.tensor([src, dst], dtype=torch.long).to(device)

print(f"edge_index: {tuple(edge_index.shape)}")

model = PyGNode2Vec(
    edge_index,
    embedding_dim=DIMENSIONS,
    walk_length=WALK_LENGTH,
    context_size=WINDOW,
    walks_per_node=NUM_WALKS,
    num_negative_samples=1,
    p=P, q=Q,
    sparse=True,
).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"PyG Node2Vec model: dim={DIMENSIONS}, params={n_params:,}")


edge_index: (2, 4138234)


PyG Node2Vec model: dim=64, params=2,887,296


In [5]:
loader = model.loader(batch_size=128, shuffle=True, num_workers=0)
optimizer = torch.optim.SparseAdam(list(model.parameters()), lr=0.01)

EPOCHS = 5   # PyG walks over all nodes per epoch; 5 is enough for stable embeddings
loss_history = []

t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for pos_rw, neg_rw in loader:
        pos_rw, neg_rw = pos_rw.to(device), neg_rw.to(device)
        optimizer.zero_grad()
        loss = model.loss(pos_rw, neg_rw)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    avg = total_loss / max(n_batches, 1)
    loss_history.append(avg)
    print(f"  epoch {epoch:d}/{EPOCHS}  avg loss={avg:.4f}  ({time.time() - t0:.1f}s elapsed)")
train_seconds = time.time() - t0
print(f"\nV3 training done in {train_seconds:.1f}s")


  epoch 1/5  avg loss=3.9388  (27.2s elapsed)


  epoch 2/5  avg loss=1.6305  (48.5s elapsed)


  epoch 3/5  avg loss=1.2043  (71.2s elapsed)


  epoch 4/5  avg loss=1.0943  (93.1s elapsed)


  epoch 5/5  avg loss=1.0595  (115.5s elapsed)

V3 training done in 115.5s


In [6]:
# Extract embeddings and cache in the same format nb07 uses
model.eval()
with torch.no_grad():
    emb_tensor = model.embedding.weight.detach().cpu().numpy()
emb = {n: emb_tensor[i] for i, n in enumerate(node_list)}

cache_key = f"V3_pyg_d{DIMENSIONS}_p{P}_q{Q}_wl{WALK_LENGTH}_nw{NUM_WALKS}_w{WINDOW}_seed{SEED}"
with open(CACHE / f"{cache_key}.pkl", "wb") as f:
    pickle.dump(emb, f)
print(f"Cached: {cache_key}.pkl ({len(emb):,} embeddings)")


Cached: V3_pyg_d64_p1_q1_wl20_nw5_w10_seed42.pkl (45,114 embeddings)


## 4 · Score the locked pair universe (three operators, same as nb05/07)

In [7]:
def build_features(pairs_df, emb, op):
    d = len(next(iter(emb.values())))
    out_dim = d if op == "hadamard" else 2 * d
    X = np.zeros((len(pairs_df), out_dim), dtype=np.float32)
    have = np.zeros(len(pairs_df), dtype=bool)
    for i, (c, dd) in enumerate(zip(pairs_df["compound"].to_numpy(),
                                    pairs_df["disease"].to_numpy())):
        u, v = ("Compound", c), ("Disease", dd)
        if u in emb and v in emb:
            eu, ev = emb[u], emb[v]
            X[i]    = eu * ev if op == "hadamard" else np.concatenate([eu, ev])
            have[i] = True
    return X, have


def cosine_scores(pairs_df, emb):
    norms = {n: e / (np.linalg.norm(e) + 1e-12) for n, e in emb.items()}
    s = np.zeros(len(pairs_df))
    for i, (c, dd) in enumerate(zip(pairs_df["compound"].to_numpy(),
                                    pairs_df["disease"].to_numpy())):
        u, v = ("Compound", c), ("Disease", dd)
        if u in norms and v in norms:
            s[i] = float(norms[u] @ norms[v])
    return s


def score_lr(X, y, idx_train, idx_test):
    clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0, random_state=SEED)
    clf.fit(X[idx_train], y[idx_train])
    return clf.predict_proba(X)[:, 1]


def bootstrap_ci(y_true, y_score, metric, B=500, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, n, n)
        if y_true[idx].sum() == 0:
            vals[b] = np.nan; continue
        vals[b] = metric(y_true[idx], y_score[idx])
    vals = vals[~np.isnan(vals)]
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


X_had, have    = build_features(pairs, emb, "hadamard")
X_cat, _       = build_features(pairs, emb, "concat")
s_cos          = cosine_scores(pairs, emb)
s_had          = score_lr(X_had, y, idx_train, idx_test)
s_cat          = score_lr(X_cat, y, idx_train, idx_test)

y_test = y[idx_test]

rows = []
for name, s in [("hadamard+LR", s_had), ("concat+LR", s_cat), ("cosine", s_cos)]:
    st = s[idx_test]
    auroc = float(roc_auc_score(y_test, st))
    auprc = float(average_precision_score(y_test, st))
    lo_roc, hi_roc = bootstrap_ci(y_test, st, roc_auc_score)
    lo_pr,  hi_pr  = bootstrap_ci(y_test, st, average_precision_score)
    rows.append({
        "variant": "V3 (PyG, full Hetionet)",
        "method": name,
        "AUROC": round(auroc, 4),
        "AUROC_95%": f"[{lo_roc:.3f}, {hi_roc:.3f}]",
        "AUPRC": round(auprc, 4),
        "AUPRC_95%": f"[{lo_pr:.4f}, {hi_pr:.4f}]",
        "pair_coverage": round(float(have.mean()), 3),
    })
v3_results = pd.DataFrame(rows)
v3_results


,variant,method,AUROC,AUROC_95%,AUPRC,AUPRC_95%,pair_coverage
0,"V3 (PyG, full Hetionet)",hadamard+LR,0.8980,"[0.877, 0.918]",0.0551,"[0.0355, 0.0904]",1.0
1,"V3 (PyG, full Hetionet)",concat+LR,0.9110,"[0.891, 0.930]",0.0514,"[0.0346, 0.0826]",1.0
2,"V3 (PyG, full Hetionet)",cosine,0.8697,"[0.848, 0.890]",0.0222,"[0.0170, 0.0297]",1.0


## 5 · Compare with nb07's V1 / V2 / V4

In [8]:
ablation = json.loads((ART / "predictions" / "graph_ablation_meta.json").read_text())
prior_rows = pd.DataFrame(ablation["results"])
combined = pd.concat([prior_rows, v3_results], ignore_index=True)

# Concat+LR view (the headline)
head = combined[combined["method"] == "concat+LR"][["variant", "AUROC", "AUROC_95%", "AUPRC", "pair_coverage"]]
print("=== Concat+LR across all variants ===")
print(head.to_string(index=False))


=== Concat+LR across all variants ===
                variant  AUROC      AUROC_95%  AUPRC  pair_coverage
                     V1 0.7804 [0.743, 0.821] 0.0396          0.086
                     V2 0.8991 [0.876, 0.922] 0.0484          1.000
                     V4 0.8967 [0.874, 0.919] 0.0457          0.894
V3 (PyG, full Hetionet) 0.9110 [0.891, 0.930] 0.0514          1.000


## 6 · Persist

Append V3 results to `graph_ablation_meta.json` so downstream analyses see the complete four-variant table.

In [9]:
updated = dict(ablation)
updated["results"] = updated["results"] + [r for r in rows]
updated["v3_supplement"] = {
    "produced_by": "07b_v3_pyg.ipynb",
    "library": f"torch {torch.__version__} + torch_geometric (MPS Node2Vec)",
    "config": {
        "dimensions": DIMENSIONS, "p": P, "q": Q,
        "walk_length": WALK_LENGTH, "num_walks": NUM_WALKS, "window": WINDOW,
        "hub_K": HUB_K, "seed": SEED, "device": str(device),
        "train_seconds": round(train_seconds, 1),
        "cache_key": cache_key,
    },
    "walk_graph": {
        "kinds_kept": "all 11",
        "n_nodes": int(G.number_of_nodes()),
        "n_edges": int(G.number_of_edges()),
        "test_ctd_edges_removed": int(dropped),
    },
    "final_loss": round(loss_history[-1], 4),
}
(ART / "predictions" / "graph_ablation_meta.json").write_text(json.dumps(updated, indent=2, default=str))
print("Updated: artifacts/predictions/graph_ablation_meta.json (V3 rows appended)")


Updated: artifacts/predictions/graph_ablation_meta.json (V3 rows appended)
